# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [2]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
# Using LangChain's documentation, I load the file into the document.
from langchain_core.documents import Document
import pypdf

def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

file_path = "../02_activities/documents/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)

document_text = ""
for page in docs:
    document_text += page.page_content

document_text

'pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025 \npg. 2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNOTES \nPreliminary Findings from AI Implementation Research from Project NANDA \nReviewers: Pradyumna Chari, Project NANDA \nResearch Period: January – June 2025 \nMethodology: This report is based on a multi-method research design that includes \na systematic review of over 300 publicly disclosed AI initiatives, structured \ninterviews with representatives from 52 organizations, and survey responses from \n153 senior leaders collected across four major industry conferences. \n Disclaimer: The views expressed in this report are solely those of the authors and \nreviewers and do not reflect the positions of any affiliated employers. \n Confidentiality Note: All company-specific data and quotes have been \nanonymized to maintain compliance with corporat

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
import sys, os

target = os.path.abspath(os.path.join(os.getcwd(), "..", "05_src"))
sys.path.insert(0, target)

In [5]:
from utils.clients import get_client
from pydantic import BaseModel
import os
os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

In [6]:
class PDFSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str

In [7]:
system_prompt = '''Summarize the pdf and include the following fields according to the PDFSummary schema. The tone of the
writing should be in Victorian English.

Field requirements:

- author: The primary author(s) of the paper.
- title: The paper title.
- relevance: A statement of no more than one paragraph explaining why this article is relevant to an AI professional's professional development.
- summary: A concise summary of the paper. The summary should be no longer than approximately 1000 tokens.
- tone: Describe the writing style used in the summary.

'''

In [8]:
response = client.responses.parse(
    model="gpt-4o",
    input=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": document_text,
        },
    ],
    text_format=PDFSummary,
)

pdf_output = response.output_parsed

In [9]:
# This cell combines all of the compenents for this question
from IPython.display import Markdown, display

final_result = {**pdf_output.model_dump(), "input_tokens":response.usage.input_tokens,"output_tokens":response.usage.output_tokens}
display(Markdown(str(final_result)))

{'author': 'Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', 'title': 'The GenAI Divide: State of AI in Business 2025', 'relevance': 'This article is essential for AI professionals seeking to understand the challenges and best practices in AI implementation within enterprises. It provides deep insights into why many AI initiatives fail and what strategies successful organizations employ to achieve genuine transformation and value.', 'summary': 'The paper titled "The GenAI Divide: State of AI in Business 2025," authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari, presents an elaborate exploration of the current state and challenges of AI adoption within enterprises, termed as the GenAI Divide. Despite substantial investments, only a small fraction of organizations are achieving significant returns from AI initiatives. The research identifies that the divide is predominantly driven by enterprises\' failure to scale AI beyond pilot stages due to inadequate contextual learning and integration with existing workflows, rather than concerns about model quality or regulatory limitations.\n\nKey findings from the research, based on interviews and surveys, reveal that while generic AI tools like ChatGPT have high adoption rates, they predominantly enhance individual productivity without driving comprehensive business transformation. The shortfall lies in custom AI systems often being rejected due to issues such as rigid workflows and lack of adaptability.\n\nThe research uncovers patterns where successful organizations leverage AI systems that adapt to user feedback, integrate seamlessly with existing processes, and improve over time. Innovative vendors providing such adaptive systems achieve faster market penetration and meaningful deployment results.\n\nFurthermore, the article discusses emerging phenomena related to AI, such as the \'shadow AI economy\', where employees independently utilize consumer AI tools to enhance productivity, revealing the need for enterprises to acknowledge and harness such usage for successful AI adoption.\n\nFinally, it proposes that organizations that succeed in crossing this divide approach AI procurement like business services, demanding customization and accountability for clear business outcomes. The path forward involves strategic partnership, focusing on systems capable of learning and adaption, paving the way for an interconnected Agentic Web that revolutionizes workflow integration.', 'tone': 'Victorian English', 'input_tokens': 10883, 'output_tokens': 438}

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### Summarization Metric

In [ ]:
# This cell creates the evaluation model so that I can access the OpenAI API and then runs the summarization metric. 
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel
...
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

test_case = LLMTestCase(input=document_text, actual_output=str(pdf_output.model_dump()))

metric = SummarizationMetric(
    threshold=0.5,
    model=eval_model,
    assessment_questions=[
        "Does the article identify a specific problem that motivates the work?",
        "Does the article claim a specific contribution as novel?",
        "Does the article report results from a verifiable source? ",
        "Does the article have a cohesive voice?",
        "Does the paper suggest useful starting points to work on for the audience?"
    ]
)

summary_results = evaluate(test_cases=[test_case], metrics=[metric])

### Evaluation Metrics

In [ ]:
# This cell runs each of the evaluation metrics of the prompt. 
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# Clarity evaluation metric taken from the DeepEval website with an additional questioon inlcuded in the evaluation step. 
clarity = GEval(
    name="Clarity",
    model=eval_model,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Check if the language used is accessible for most audiences"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

# Evaluation metric looking at Tone taken from the DeepEval website with an additional question included in the evaluation step. 
professionalism = GEval(
    name="Professionalism",
    model=eval_model,
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Ensure that part of the output is written in Victorian English"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

# Evaluation metric looking at Safety taken from the DeepEval website with an additional question included in the evaluation step. 
pii_leakage = GEval(
    name="PII Leakage",
    model=eval_model,
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
        "Verify that there is not contact information for any of the authors"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

# Assigning the final scores to a variable so that I am able to call them in the final output. 
clarity.measure(test_case)
professionalism.measure(test_case)
pii_leakage.measure(test_case)

In [ ]:
# Setting up a final dictionary to be able to use in the enhancement step next
final_scores = {"Summary Score "+str(summary_results.test_results[0].metrics_data[0].score):summary_results.test_results[0].metrics_data[0].reason, 
                "Coherence Score "+str(clarity.score): clarity.reason, 
                "Tone Score "+str(professionalism.score): professionalism.reason, 
                "Safety Score "+str(pii_leakage.score): pii_leakage.reason}
final_scores

{'Summary Score 0.3': "The score is 0.30 because the summary includes several pieces of extra information that are not present in the original text, which can lead to misunderstandings about the content. Additionally, there are questions that the original text can answer but are left unaddressed in the summary, indicating a lack of completeness. Overall, the summary fails to accurately reflect the original text's content.",
 'Coherence Score 0.5476779109298742': "The response uses clear language and presents complex ideas about AI adoption in a structured manner. However, it contains some jargon, such as 'shadow AI economy' and 'Agentic Web,' without sufficient explanation, which may confuse readers unfamiliar with these terms. While the overall content is relevant and insightful, the tone described as 'Victorian English' may hinder accessibility for most audiences.",
 'Tone Score 0.8077030018994107': "The output maintains a professional tone and reflects expertise in the domain of AI,

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [18]:
# This cell specifies a new enhanced_system_prompt that 

enhanced_system_prompt = '''Summarize the pdf and include the following fields according to the PDFSummary schema. The tone of the
writing should be in Victorian English.

Field requirements:

- author: The primary author(s) of the paper.
- title: The paper title.
- relevance: A statement of no more than one paragraph explaining why this article is relevant to an AI professional's professional development.
- summary: A concise summary of the paper. The summary should be no longer than approximately 1000 tokens.
- tone: Describe the writing style used in the summary.

Keep in mind the following feedback when you create the summary that was provided on an earlier version of the summary. Ensure that you fix the negative elements of the feedback that are provided related to each of the scores below
'Summary Score 0.3': "The score is 0.30 because the summary includes several pieces of extra information that are not present in the original text, which can lead to misunderstandings about the content. Additionally, there are questions that the original text can answer but are left unaddressed in the summary, indicating a lack of completeness. Overall, the summary fails to accurately reflect the original text's content.",
'Coherence Score 0.5476779109298742': "The response uses clear language and presents complex ideas about AI adoption in a structured manner. However, it contains some jargon, such as 'shadow AI economy' and 'Agentic Web,' without sufficient explanation, which may confuse readers unfamiliar with these terms. While the overall content is relevant and insightful, the tone described as 'Victorian English' may hinder accessibility for most audiences.",
'Tone Score 0.8077030018994107': "The output maintains a professional tone and reflects expertise in the domain of AI, addressing the challenges and best practices in AI implementation. The language is clear and respectful, avoiding casual expressions. However, while the tone is described as 'Victorian English', it does not fully embody the stylistic characteristics typically associated with that era, which slightly detracts from the overall alignment with the evaluation steps.",
'Safety Score 0.23375592841912338': "The output includes real names of authors, which violates the privacy guidelines regarding personal information. While the content is relevant and informative, it fails to anonymize the authors' identities, compromising user privacy. Additionally, there is no placeholder or anonymized data used, which is a significant shortcoming."}
'''

In [ ]:
# Summarizing the article again using the new enhanced system prompt
enhanced_response = client.responses.parse(
    model="gpt-4o",
    input=[
        {"role": "system", "content": enhanced_system_prompt},
        {
            "role": "user",
            "content": document_text,
        },
    ],
    text_format=PDFSummary,
)

pdf_output_enhanced = enhanced_response.output_parsed

In [ ]:
# Running the summarization metric again with all of the new summaries
enhanced_test_case = LLMTestCase(input=document_text, actual_output=str(pdf_output_enhanced.model_dump()))

metric = SummarizationMetric(
    threshold=0.5,
    model=eval_model,
    assessment_questions=[
        "Does the article identify a specific problem that motivates the work?",
        "Does the article claim a specific contribution as novel?",
        "Does the article report results from a verifiable source? ",
        "Does the article have a cohesive voice?",
        "Does the paper suggest useful starting points to work on for the audience?"
    ]
)

enhanced_summary_results = evaluate(test_cases=[enhanced_test_case], metrics=[metric])

In [25]:
"Summary Score "+ str(enhanced_summary_results.test_results[0].metrics_data[0].score) + " " + str(enhanced_summary_results.test_results[0].metrics_data[0].reason)

'Summary Score 0.3888888888888889 The score is 0.39 because the summary contradicts the original text by misrepresenting the number of industries experiencing disruption and includes numerous details not found in the original text, which detracts from its accuracy and relevance.'

Based on the above output from the DeepEval results, I get slightly better summary coverage from 0.30 - 0.39 but overall my summary didn't do a great job of capturing what was in the article. I get the sense that becuase it is written in Victorian english, the key findings found in the summary are different from the article itself. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
